# Baseline LDM — perio-KPT (Google Colab)

**Run the next code cell first** (`Setup + imports`). Without it you get `No module named 'src'`.

Colab in VS Code often starts in `/content` while your repo is elsewhere — Setup searches for the folder that contains `src/`.

## Setup + imports (run this cell first)

In [16]:
!git clone https://github.com/Rafa13io/computer_vision_project.git

fatal: destination path 'computer_vision_project' already exists and is not an empty directory.


In [17]:
!ls


computer_vision_project  docs	    README.md	      scripts
configs			 notebooks  requirements.txt  src


In [18]:
!cd computer_vision_project && git switch cursor

Already on 'cursor'
Your branch is up to date with 'origin/cursor'.


In [23]:
import os
import sys
from pathlib import Path


def find_project_root() -> Path:
    """Locate repo root (directory containing src/utils/config.py)."""
    checked: set[Path] = set()
    candidates: list[Path] = []

    # Current dir and parents (notebook may live in notebooks/)
    p = Path.cwd().resolve()
    for _ in range(8):
        candidates.append(p)
        if p.parent == p:
            break
        p = p.parent

    # Common Colab locations
    candidates.extend(
        [
            Path("/content/computer_vision_project"),
            Path("/content/drive/MyDrive/computer_vision_project"),
            Path("/content/drive/MyDrive/Colab Notebooks/computer_vision_project"),
        ]
    )

    for cand in candidates:
        cand = cand.resolve()
        if cand in checked:
            continue
        checked.add(cand)
        if (cand / "src" / "utils" / "config.py").is_file():
            return cand

    raise FileNotFoundError(
        "Project root not found. On Colab, clone or upload the full repo, e.g.\n"
        "  !git clone https://github.com/Rafa13io/computer_vision_project.git\n"
        "Then re-run this cell."
    )


ROOT = find_project_root()
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("Project root:", ROOT)
print("Working directory:", Path.cwd())
print("src on path:", any("computer_vision" in x or x.endswith("/content") or "src" in x for x in sys.path[:3]))

# Dependencies
import subprocess
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements.txt")],
    check=False,
)

import torch
from src.utils.config import load_config, project_root
from src.utils.seed import set_seed

print("PyTorch:", torch.__version__, "| CUDA:", torch.cuda.is_available())

Project root: /content/computer_vision_project
Working directory: /content/computer_vision_project
src on path: True


PyTorch: 2.10.0+cu128 | CUDA: True


## Globals

In [24]:
CFG_PATH = ROOT / "configs/baseline_ldm.yaml"
cfg = load_config(CFG_PATH)
set_seed(int(cfg["seed"]))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

device: cuda


## Link perio-KPT from Google Drive (run before build_manifest)

The Zenodo dataset is **not in GitHub**. After [requesting access](https://zenodo.org/records/17272200), download the zip to Drive, extract it, then set `EXTRACTED` below.

In [ ]:
import os
from pathlib import Path

try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Not in Colab — skip drive.mount if data is already local")

# >>> EDIT: folder where you extracted the Zenodo download <<<
# Must contain: 1_Experiment/standard_box/
EXTRACTED = Path("/content/drive/MyDrive/perio-KPT")

PROJECT_DATA = ROOT / "data" / "perio-KPT"
PROJECT_DATA.parent.mkdir(parents=True, exist_ok=True)

def _valid(root: Path) -> bool:
    return (root / "1_Experiment" / "standard_box").is_dir()

if not EXTRACTED.is_dir():
    raise FileNotFoundError(
        f"Dataset not found at {EXTRACTED}\n"
        "1) Get access: https://zenodo.org/records/17272200\n"
        "2) Download zip to Google Drive and extract\n"
        "3) Set EXTRACTED to that folder (see 1_Experiment/ inside)"
    )

if not _valid(EXTRACTED):
    # handle extra folder from zip, e.g. perio-KPT-v2/perio-KPT/...
    found = list(EXTRACTED.rglob("standard_box"))
    found = [p for p in found if p.parent.name == "1_Experiment"]
    if found:
        EXTRACTED = found[0].parent.parent
        print("Detected dataset root:", EXTRACTED)
    else:
        raise FileNotFoundError(
            f"No 1_Experiment/standard_box under {EXTRACTED}. Check extract path."
        )

if PROJECT_DATA.exists() or PROJECT_DATA.is_symlink():
    print("data/perio-KPT already present:", PROJECT_DATA.resolve())
else:
    os.symlink(EXTRACTED.resolve(), PROJECT_DATA)
    print("Symlinked:", PROJECT_DATA, "->", EXTRACTED.resolve())

os.environ["PERIO_KPT_ROOT"] = str(EXTRACTED.resolve())
assert _valid(PROJECT_DATA), "Link failed — check paths"
print("Ready to run build_manifest")

## Data

1. Download [perio-KPT](https://zenodo.org/records/17272200) → `data/perio-KPT/`
2. Run the cell below (uses `ROOT` from Setup — no separate `!python` needed)

## Train

```bash
!python -m src.train.train_vae --config configs/baseline_ldm.yaml
!python -m src.train.train_ldm --config configs/baseline_ldm.yaml
```

In [25]:
!python -m src.train.train_vae --config configs/baseline_ldm.yaml
!python -m src.train.train_ldm --config configs/baseline_ldm.yaml

Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/content/computer_vision_project/src/train/train_vae.py", line 122, in <module>
    main()
  File "/content/computer_vision_project/src/train/train_vae.py", line 43, in main
    train_ds = PatchDataset(
               ^^^^^^^^^^^^^
  File "/content/computer_vision_project/src/data/patch_dataset.py", line 25, in __init__
    with open(manifest_path, encoding="utf-8") as f:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: '/content/computer_vision_project/data/processed/manifest.csv'
Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/content/computer_vision_project/src/train/train_ldm.py", line 121, in <module>
    main()
  File "/content/computer_vision_project/src/train/train_ldm.py", line 3

## Evaluation — samples & failure analysis

```bash
!python -m src.train.sample --config configs/baseline_ldm.yaml
```

Document structural errors: wrong bone level, CEJ blur, root shape, texture-only fakes.

# Build manifest (run after dataset is in data/perio-KPT/)
import subprocess

subprocess.check_call(
    [sys.executable, "-m", "src.data.build_manifest", "--config", "configs/baseline_ldm.yaml"],
    cwd=ROOT,
)



In [26]:
!python -m src.data.build_manifest --config configs/baseline_ldm.yaml


Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/content/computer_vision_project/src/data/build_manifest.py", line 213, in <module>
    main()
  File "/content/computer_vision_project/src/data/build_manifest.py", line 209, in main
    build_manifest(cfg, root)
  File "/content/computer_vision_project/src/data/build_manifest.py", line 99, in build_manifest
    raise FileNotFoundError(
FileNotFoundError: Dataset not found at /content/computer_vision_project/data/perio-KPT. Download perio-KPT from https://zenodo.org/records/17272200


In [ ]:
!python -m src.train.train_vae --config configs/baseline_ldm.yaml


In [ ]:
!python -m src.train.train_ldm --config configs/baseline_ldm.yaml


In [ ]:
!python -m src.train.sample --config configs/baseline_ldm.yaml